In [0]:
bronze_df = spark.read. format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv")

bronze_df.write.format("delta").mode("overwrite").saveAsTable("bronze_ecommerce")

In [0]:
from pyspark.sql.functions import col

silver_df = spark.table("bronze_ecommerce") \
.dropna(subset=["user_id","product_id","event_type"])

silver_df = silver_df.withColumn(
"purchase_flag",
(col("event_type") == "purchase").cast("int")
)

silver_df.write.format("delta").mode("overwrite").saveAsTable("silver_ecommerce")

In [0]:
from pyspark.sql.functions import sum, count

gold_df = spark.table("silver_ecommerce") \
    .groupBy("user_id") \
    .agg(
    sum("purchase_flag").alias("total_purchases"),
    count("product_id").alias("num_interactions")
    )

gold_df = gold_df.withColumn(
    "label",
    (gold_df.total_purchases > 0).cast("int")
)
gold_df.write.format("delta").mode("overwrite").saveAsTable("gold_user_features_1")

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

features_df = spark.table("gold_user_features_1")

assembler = VectorAssembler(
inputCols=["num_interactions"],
outputCol="features"
)

training_data = assembler.transform(features_df)

train_df, test_df = training_data.randomSplit([0.8, 0.2], seed=42)

In [0]:
lr = LogisticRegression(
featuresCol="features",
labelCol="label"
)
model = lr.fit(train_df)

In [0]:
predictions = model.transform(test_df)

predictions.select(
    "num_interactions",
    "label",
    "prediction",
    "probability"
).show()

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    metricName="areaUnderROC"
)
auc = evaluator.evaluate(predictions)

print("AUC:", auc)

In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    metricName="accuracy"
)
accuracy = accuracy_evaluator. evaluate(predictions)
print("Accuracy:", accuracy)